In [ ]:
#new cleaned given by gemini
# ==============================
# IMPORTS
# ==============================

import os
import pandas as pd
import chardet
from openpyxl import load_workbook
import csv
from google.colab import files


# ==============================
# FILE UPLOAD (COLAB)
# ==============================

uploaded = files.upload()

for fn in uploaded.keys():
    print(f'User uploaded file "{fn}" with length {len(uploaded[fn])} bytes')
    file_path = f'/content/{fn}'
    break

print(f"\nUsing uploaded file: {file_path}")


# ==============================
# SHEET SELECTION FUNCTION
# ==============================

def sheet_selection_cli(excel_data, suggested_sheet=None):

    sheet_names = list(excel_data.keys())

    if len(sheet_names) == 1:
        print(f"\nOnly one sheet found: {sheet_names[0]}")
        return excel_data[sheet_names[0]]

    print("\nAvailable Sheets:")
    for i, name in enumerate(sheet_names):
        print(f"[{i}] {name} | Shape: {excel_data[name].shape}")

    print("\nChoose an option:")
    print("1 → Use suggested sheet")
    print("2 → Select one sheet")
    print("3 → Select multiple sheets")

    choice = input("\nEnter your choice (1/2/3): ")

    # OPTION 1
    if choice == "1":
        if suggested_sheet and suggested_sheet in excel_data:
            return excel_data[suggested_sheet]
        return excel_data[sheet_names[0]]

    # OPTION 2
    elif choice == "2":
        index = int(input("Enter sheet index: "))
        return excel_data[sheet_names[index]]

    # OPTION 3
    elif choice == "3":

        indices = input("Enter sheet indices separated by comma (e.g., 0,2): ")
        indices = [int(i.strip()) for i in indices.split(",")]

        selected_dfs = {sheet_names[i]: excel_data[sheet_names[i]] for i in indices}

        print("\nWhat would you like to do?")
        print("1 → Combine sheets into one dataset")
        print("2 → Keep sheets separate")

        action_choice = input("Enter choice (1/2): ")

        # Keep separate
        if action_choice == "2":
            return selected_dfs

        # Combine
        elif action_choice == "1":

            print("\nHow should the sheets be combined?")
            print("1 → Concatenate")
            print("2 → Merge")

            combine_choice = input("Enter choice (1/2): ")

            dfs_list = list(selected_dfs.values())

            # CONCAT
            if combine_choice == "1":
                combined_df = pd.concat(dfs_list, ignore_index=True)
                print("\nSheets concatenated successfully.")
                return combined_df

            # MERGE
            elif combine_choice == "2":

                key = input("Enter column name to merge on: ")

                print("\nSelect merge type:")
                print("1 → inner")
                print("2 → left")
                print("3 → right")
                print("4 → outer")
                print("5 → cross")

                merge_type_map = {
                    "1": "inner",
                    "2": "left",
                    "3": "right",
                    "4": "outer",
                    "5": "cross"
                }

                how = merge_type_map.get(input("Enter choice (1-5): "), "inner")

                merged_df = dfs_list[0]

                for df_part in dfs_list[1:]:
                    if how == "cross":
                        merged_df = merged_df.merge(df_part, how="cross")
                    else:
                        merged_df = merged_df.merge(df_part, on=key, how=how)

                print(f"\nSheets merged successfully using '{how}' join.")
                return merged_df

    print("\nInvalid choice. Defaulting to first sheet.")
    return excel_data[sheet_names[0]]


# ==============================
# INGESTION
# ==============================

ext = os.path.splitext(file_path)[1].lower()

# CSV / TXT
if ext in [".csv", ".txt"]:

    with open(file_path, 'rb') as f:
        result = chardet.detect(f.read(10000))
    encoding = result['encoding']

    with open(file_path, 'r', encoding=encoding) as f:
        sample = f.read(5000)
        delimiter = csv.Sniffer().sniff(sample).delimiter

    df = pd.read_csv(file_path, encoding=encoding, delimiter=delimiter)


# EXCEL
elif ext in [".xls", ".xlsx"]:

    xls = pd.ExcelFile(file_path, engine="openpyxl")
    wb = load_workbook(file_path, data_only=True)

    sheet_names = xls.sheet_names

    print("\n========== EXCEL SHEET ANALYSIS ==========\n")

    valid_sheets = {}
    sheet_report = {}

    for sheet in sheet_names:
        temp_df = xls.parse(sheet)
        ws = wb[sheet]

        rows, cols = temp_df.shape
        non_null = temp_df.notna().sum().sum()
        total_cells = rows * cols
        null_ratio = 1 - (non_null / total_cells) if total_cells > 0 else 1

        numeric_cols = temp_df.select_dtypes(include='number').shape[1]
        object_cols = temp_df.select_dtypes(include='object').shape[1]

        # Detect charts
        has_chart = len(ws._charts) > 0

        # Detect merged cells
        merged_cells = len(ws.merged_cells.ranges)

        memory_usage = round(temp_df.memory_usage(deep=True).sum() / 1024, 2)

        sheet_report[sheet] = {
            "rows": rows,
            "columns": cols,
            "null_ratio": round(null_ratio, 3),
            "numeric_columns": numeric_cols,
            "text_columns": object_cols,
            "charts_detected": has_chart,
            "merged_cell_ranges": merged_cells,
            "memory_kb": memory_usage
        }

        # Validation logic
        if (
            cols >= 3
            and rows > 5
            and null_ratio < 0.8
            and not has_chart
        ):
            valid_sheets[sheet] = temp_df

    if len(valid_sheets) == 1:
       chosen_sheet = list(valid_sheets.keys())[0]
       chosen_df = valid_sheets[chosen_sheet]

       print(f"\n✅ Chosen Sheet: {chosen_sheet}")
       print("\nPreview (First 5 Rows):")
       print(chosen_df.head())
       print("Shape:", chosen_df.shape)


    elif len(valid_sheets) > 1:
       print("\n⚠ Multiple valid sheets detected:\n")

       for name, data in valid_sheets.items():
           print(f"\n--- Sheet: {name} ---")
           print("Preview (First 5 Rows):")
           print(data.head())
           print("Shape:", data.shape)


    else:
        print("\n❌ No valid sheets found.")

    # Print report
    for name, info in sheet_report.items():
        print(f"\n--- Sheet: {name} ---")
        for key, value in info.items():
            print(f"{key}: {value}")

        if name in valid_sheets:
            print("Status: ✅ Likely Data Sheet")
        else:
            print("Status: ❌ Rejected")

    print("\n==========================================\n")

    if len(valid_sheets) == 1:
        df = list(valid_sheets.values())[0]
    elif len(valid_sheets) > 1:
        df = valid_sheets
    else:
        raise ValueError("No valid data sheets found.")


# JSON
elif ext == ".json":

    df = pd.read_json(file_path)

else:
    raise ValueError("Unsupported file type")


# ==============================
# SHEET SELECTION (ONLY FOR EXCEL)
# ==============================

if ext in [".xls", ".xlsx"]:

    print("\n========== SHEET SELECTION PROCESS ==========\n")

    excel_data = {
        sheet: xls.parse(sheet)
        for sheet in sheet_names
    }

    working_df = sheet_selection_cli(excel_data)

else:
    working_df = df


# ==============================
# 🧹 GLOBAL NAN & BLANK CLEANUP
# ==============================

# This line removes any rows that are completely empty
if isinstance(working_df, dict):
    for name in working_df:
        working_df[name] = working_df[name].dropna(how='all').reset_index(drop=True)
        print(f"🧹 Cleaned empty rows in sheet: {name}")
else:
    working_df = working_df.dropna(how='all').reset_index(drop=True)
    print("🧹 Global row cleanup: Removed completely empty rows.")

# ==============================
# text in numerical column analysis (Refined)
# ==============================

def clean_mixed_numeric_logic(df, threshold=0.8):
    """
    Targets columns that are mostly numeric but 'polluted' with text.
    Goal: Transform columns from <100% numeric to 100% numeric.
    """
    for col in df.columns:
        # Attempt to convert to numeric; text strings become NaN
        numeric_series = pd.to_numeric(df[col], errors='coerce')

        valid_count = numeric_series.notnull().sum()
        total_count = len(df[col])
        ratio = valid_count / total_count if total_count > 0 else 0

        # 1. Identify if it's a Numeric Column (based on your 80% threshold)
        if ratio >= threshold:

            # 2. If it's LESS than 100%, it means there is categorical noise to remove
            if ratio < 1.0:
                noise_count = total_count - valid_count
                print(f"📍 Recovering Numeric Column: '{col}'")
                print(f"   -> Found {noise_count} categorical/messy values ({ratio:.1%} numeric).")

                # Calculate mean from the valid numbers
                mean_val = numeric_series.mean()

                # Fill the 'noise' holes with the average
                df[col] = numeric_series.fillna(mean_val)
                print(f"   -> Result: Column is now 100% numeric (Mean {mean_val:.2f} applied).")

            else:
                # If it's already 100% numeric, just ensure it's the correct data type
                df[col] = numeric_series

    return df

# Automatically run the analysis on your selected dataset
if isinstance(working_df, dict):
    for name in working_df:
        working_df[name] = clean_mixed_numeric_logic(working_df[name])
else:
    working_df = clean_mixed_numeric_logic(working_df)

# ==============================
# 🗂️ CATEGORICAL VALUE CLEANUP
# ==============================

def clean_categorical_logic(df):
    """
    Identifies categorical columns and replaces NaNs or 'UNKNOWN'
    placeholders with the most frequent value (Mode).
    """
    cat_cols = df.select_dtypes(include=['object']).columns

    for col in cat_cols:
        # Count original NaNs and "UNKNOWN" placeholders
        unknown_mask = df[col].astype(str).str.upper().isin(['UNKNOWN', 'ERROR', 'NAN', 'NONE', ''])
        missing_count = unknown_mask.sum()

        if missing_count > 0:
            print(f"📍 Cleaning Categorical Column: '{col}'")
            print(f"   -> Found {missing_count} missing/placeholder values.")

            # Identify the Mode (most frequent valid value)
            valid_values = df.loc[~unknown_mask, col]
            if not valid_values.empty:
                mode_val = valid_values.mode()[0]

                # Replace placeholders and NaNs with the Mode
                df.loc[unknown_mask, col] = mode_val
                print(f"   -> Result: Replaced placeholders with Mode: '{mode_val}'.")
            else:
                print("   -> Skip: No valid data found in column to calculate mode.")

    return df

# Run the categorical cleanup
if isinstance(working_df, dict):
    for name in working_df:
        working_df[name] = clean_categorical_logic(working_df[name])
else:
    working_df = clean_categorical_logic(working_df)


# ==============================
# FINAL PREVIEW
# ==============================

print("\n========== FINAL DATA PREVIEW ==========\n")

if isinstance(working_df, dict):
    for name, data in working_df.items():
        print(f"\n--- Sheet: {name} ---")
        print(data.head())
        print("Shape:", data.shape)
else:
    print(working_df.head())
    print("Shape:", working_df.shape)

# ==============================
# 📥 EXPORT CLEANED DATA TO EXCEL
# ==============================

def export_to_excel(df, filename="Cleaned_Dataset.xlsx"):
    """
    Saves the cleaned dataframe to an Excel file and triggers a local download.
    """
    # Create the Excel file on the Colab server
    df.to_excel(filename, index=False)
    print(f"📊 Dataset successfully converted to {filename}")

    # Trigger the download to your local computer
    from google.colab import files
    files.download(filename)
    print("🚀 Download triggered. Please check your browser's download folder.")

# Run the export
if isinstance(working_df, dict):
    # If you have multiple sheets, this saves the first one by default
    # Or you can modify this to save each sheet as a separate tab
    first_sheet_name = list(working_df.keys())[0]
    export_to_excel(working_df[first_sheet_name], filename="Cleaned_MultiSheet_Output.xlsx")
else:
    export_to_excel(working_df, filename="Arun_Final_Cleaned_Data.xlsx")

Saving dirty_cafe_sales (1) (1).xlsx to dirty_cafe_sales (1) (1) (4).xlsx
User uploaded file "dirty_cafe_sales (1) (1) (4).xlsx" with length 468236 bytes

Using uploaded file: /content/dirty_cafe_sales (1) (1) (4).xlsx

========== EXCEL SHEET ANALYSIS ==========


✅ Chosen Sheet: dirty_cafe_sales (1)

Preview (First 5 Rows):
  Transaction ID    Item Quantity Price Per Unit Total Spent  Payment Method  \
0    TXN_1961373  Coffee        2              2           4     Credit Card   
1    TXN_4977031    Cake        4              3          12            Cash   
2    TXN_4271903  Cookie        4              1       ERROR     Credit Card   
3    TXN_7034554   Salad        2              5          10         UNKNOWN   
4    TXN_3160411  Coffee        2              2           4  Digital Wallet   

   Location     Transaction Date  
0  Takeaway  2023-09-08 00:00:00  
1  In-store  2023-05-16 00:00:00  
2  In-store  2023-07-19 00:00:00  
3   UNKNOWN  2023-04-27 00:00:00  
4  In-store  2023

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

🚀 Download triggered. Please check your browser's download folder.
